# 3 — Enhanced Classifier Extraction

Uses Azure AI Content Understanding's **Enhanced Classifier** pattern:
1. Deploy a custom **FS extraction analyzer** (reuses the `sec_financial_tables_v1` schema)
2. Deploy an **enhanced classifier** with `enableSegment: true` and 6 content categories
   (5 FS types + Other), each FS category linked to the extraction analyzer
3. Process each PDF in a single call — CU segments, classifies, and extracts automatically

No manual page detection required.

In [2]:
import json, os, re, time
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import ClientSecretCredential
from azure.ai.contentunderstanding import ContentUnderstandingClient

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(REPO_ROOT / ".env")

endpoint = os.environ["FOUNDRY_ENDPOINT"].split("/api/projects/")[0].rstrip("/") + "/"
credential = ClientSecretCredential(
    tenant_id=os.environ["AZURE_TENANT_ID"],
    client_id=os.environ["AZURE_CLIENT_ID"],
    client_secret=os.environ["AZURE_CLIENT_SECRET"],
)
client = ContentUnderstandingClient(endpoint=endpoint, credential=credential, api_version="2025-11-01")
print("Client ready:", endpoint)

Client ready: https://ais-aidemos-dev-01.services.ai.azure.com/


## Deploy Analyzer & Classifier

In [3]:
# Deploy the financial-table extraction analyzer (reuse existing schema)
ANALYZER_ID = "sec_financial_tables_v1"
template_path = REPO_ROOT / "analyzers" / f"{ANALYZER_ID}.json"
tmpl = json.loads(template_path.read_text(encoding="utf-8"))
tmpl["models"] = {
    "completion": os.environ["GPT41_MODEL_DEPLOYMENT"],
    "embedding": os.environ["EMBEDDING_MODEL_DEPLOYMENT"],
}

t0 = time.time()
client.begin_create_analyzer(analyzer_id=ANALYZER_ID, resource=tmpl, allow_replace=True).result()
print(f"Analyzer '{ANALYZER_ID}' deployed in {time.time() - t0:.1f}s")

Analyzer 'sec_financial_tables_v1' deployed in 5.7s


In [4]:
# Deploy the enhanced classifier with per-FS-type categories
CLASSIFIER_ID = "sec_fs_classifier_v1"

classifier_template = {
    "baseAnalyzerId": "prebuilt-document",
    "description": (
        "SEC 10-K/10-Q financial statement classifier. "
        "Segments filings into the five primary consolidated financial statements "
        "and routes each to a custom extraction analyzer."
    ),
    "config": {
        "returnDetails": True,
        "enableSegment": True,
        "contentCategories": {
            "BalanceSheet": {
                "description": (
                    "Consolidated Balance Sheets or Statement of Financial Position. "
                    "Shows total assets, total liabilities, and stockholders' equity "
                    "at specific reporting dates. Typically titled "
                    "'Consolidated Balance Sheets'."
                ),
                "analyzerId": ANALYZER_ID,
            },
            "IncomeStatement": {
                "description": (
                    "Consolidated Statements of Operations, Statements of Income, "
                    "or Income Statements. Shows revenues, costs, expenses, and "
                    "net income/loss over reporting periods. "
                    "Does NOT include Statements of Comprehensive Income."
                ),
                "analyzerId": ANALYZER_ID,
            },
            "ComprehensiveIncome": {
                "description": (
                    "Consolidated Statements of Comprehensive Income (Loss). "
                    "Shows net income plus other comprehensive income items such as "
                    "unrealized gains/losses, foreign currency translation, "
                    "and pension adjustments."
                ),
                "analyzerId": ANALYZER_ID,
            },
            "Equity": {
                "description": (
                    "Consolidated Statements of Changes in Stockholders' Equity. "
                    "Shows changes in equity components (common stock, APIC, "
                    "retained earnings, AOCI, treasury stock) across periods. "
                    "May contain multiple sub-tables for different comparison periods."
                ),
                "analyzerId": ANALYZER_ID,
            },
            "CashFlow": {
                "description": (
                    "Consolidated Statements of Cash Flows. "
                    "Shows operating, investing, and financing activities "
                    "with beginning and ending cash balances."
                ),
                "analyzerId": ANALYZER_ID,
            },
            "Other": {
                "description": (
                    "Any page that is NOT one of the five primary financial "
                    "statement tables. Includes: cover pages, table of contents, "
                    "MD&A, notes to financial statements, auditor reports, "
                    "exhibits, and all other content."
                ),
            },
        },
    },
    "models": {
        "completion": os.environ["GPT41_MODEL_DEPLOYMENT"],
        "embedding": os.environ["EMBEDDING_MODEL_DEPLOYMENT"],
    },
    "tags": {"purpose": "sec-fs-classifier", "version": "v1"},
}

t0 = time.time()
client.begin_create_analyzer(
    analyzer_id=CLASSIFIER_ID, resource=classifier_template, allow_replace=True
).result()
print(f"Classifier '{CLASSIFIER_ID}' deployed in {time.time() - t0:.1f}s")

Classifier 'sec_fs_classifier_v1' deployed in 0.6s


In [5]:
# Test run: only TRNO and Safehold to verify ordering fixes
test_names = ["trno_10Q_Q3_2024", "Safehold 2024 10-K (1) 1"]
pdf_dir = REPO_ROOT / "email" / "attachements"
test_pdfs = [p for p in sorted(pdf_dir.glob("*.pdf")) if p.stem in test_names]
print(f"Testing {len(test_pdfs)} PDFs\n")

test_results = {}
for pdf in test_pdfs:
    print(f"  {pdf.name} ...", end=" ", flush=True)
    t0 = time.time()
    poller = client.begin_analyze_binary(
        analyzer_id=CLASSIFIER_ID,
        binary_input=pdf.read_bytes(),
        content_type="application/pdf",
    )
    result = poller.result().as_dict()
    test_results[pdf.stem] = result
    elapsed = time.time() - t0

    contents = result.get("contents", [])
    cats = {}
    for seg in contents:
        cat = seg.get("category", "Unknown")
        cats[cat] = cats.get(cat, 0) + 1
    cat_str = ", ".join(f"{k}:{v}" for k, v in sorted(cats.items()))
    print(f"{len(contents)} segments, {elapsed:.1f}s  [{cat_str}]")

print("\nDone.")

Testing 2 PDFs

  Safehold 2024 10-K (1) 1.pdf ... 6 segments, 616.8s  [BalanceSheet:1, CashFlow:1, ComprehensiveIncome:1, Equity:1, IncomeStatement:1, Unknown:1]
  trno_10Q_Q3_2024.pdf ... 5 segments, 676.5s  [BalanceSheet:1, CashFlow:1, Equity:1, IncomeStatement:1, Unknown:1]

Done.


In [22]:
# ── Full 5-PDF Audit against Markdown Ground Truth ────────────────────────────
import importlib, sys, re
sys.path.insert(0, str(REPO_ROOT / "scripts"))
if "export_to_excel" in sys.modules:
    importlib.reload(sys.modules["export_to_excel"])
from export_to_excel import load_document

out_dir = REPO_ROOT / "output"
md_dir = out_dir / "markdown"

def extract_html_table(md_text, caption_pattern):
    pat = re.compile(r"<caption>[^<]*" + caption_pattern + r"[^<]*</caption>\s*(.*?)</table>", re.DOTALL | re.IGNORECASE)
    m = pat.search(md_text)
    if m: return m.group(0)
    pat2 = re.compile(caption_pattern + r".*?(<table>.*?</table>)", re.DOTALL | re.IGNORECASE)
    m2 = pat2.search(md_text)
    return m2.group(1) if m2 else None

def get_md_row_labels(table_html):
    if not table_html: return []
    rows = re.findall(r"<tr>\s*<td[^>]*>(.*?)</td>", table_html, re.DOTALL)
    return [re.sub(r"\s+", " ", re.sub(r"<[^>]+>", " ", td)).strip() for td in rows if re.sub(r"<[^>]+>", " ", td).strip()]

def fuzzy_match(a, b, n=12):
    norm = lambda s: s[:n].lower().replace(" ","").replace("-","").replace("(","").replace(")","")
    return norm(a) == norm(b)

AUDIT_MAP = {
    "10K-GOOG-01-31-2024": {"md": "10K-GOOG-01-31-2024.md", "BalanceSheet": r"CONSOLIDATED BALANCE SHEETS", "IncomeStatement": r"CONSOLIDATED STATEMENTS OF INCOME", "ComprehensiveIncome": r"CONSOLIDATED STATEMENTS OF COMPREHENSIVE INCOME", "Equity": r"CONSOLIDATED STATEMENTS OF STOCKHOLDERS.*EQUITY", "CashFlow": r"CONSOLIDATED STATEMENTS OF CASH FLOWS"},
    "10K-MSFT-07-27-2023": {"md": "10K-MSFT-07-27-2023.md", "BalanceSheet": r"BALANCE SHEETS", "IncomeStatement": r"INCOME STATEMENTS", "ComprehensiveIncome": r"COMPREHENSIVE INCOME STATEMENTS", "Equity": r"STOCKHOLDERS.*EQUITY STATEMENTS", "CashFlow": r"CASH FLOWS STATEMENTS"},
    "Safehold 2024 10-K (1) 1": {"md": "Safehold 2024 10-K (1) 1.md", "BalanceSheet": r"Consolidated Balance Sheets", "IncomeStatement": r"Consolidated Statements of Operations", "ComprehensiveIncome": r"Consolidated Statements of Comprehensive", "Equity": r"Consolidated Statements of Changes in Equity", "CashFlow": r"Consolidated Statements of Cash Flows"},
    "bnl_10K_2024": {"md": "bnl_10K_2024.md", "BalanceSheet": r"Consolidated Balance Sheets", "IncomeStatement": r"Consolidated Statements of Income", "Equity": r"Consolidated Statements of Equity", "CashFlow": r"Consolidated Statements of Cash Flows"},
    "trno_10Q_Q3_2024": {"md": "trno_10Q_Q3_2024.md", "BalanceSheet": r"Consolidated Balance Sheets", "IncomeStatement": r"Consolidated Statements of Operations", "Equity": r"Consolidated Statements of Equity", "CashFlow": r"Consolidated Statements of Cash Flows"},
}

total_pass = total_fail = 0
failures = []
ok_count = 0

for stem, cfg in AUDIT_MAP.items():
    json_path = out_dir / f"{stem}_classified.json"
    if not json_path.exists(): continue
    md_text = (md_dir / cfg["md"]).read_text(encoding="utf-8")
    tables = load_document(json_path)
    by_type = {}
    for tbl in tables:
        by_type.setdefault(tbl["statementType"], []).append(tbl)

    for fs_type, caption_pat in cfg.items():
        if fs_type == "md": continue
        ext_tables = by_type.get(fs_type, [])
        if not ext_tables:
            total_fail += 1
            failures.append(f"{stem}/{fs_type}: NOT FOUND")
            continue
        tbl = ext_tables[0]
        checks = []
        empty_lvl = sum(1 for r in tbl["rows"] if r["level"] is None or r["level"] < 0)
        checks.append(("levels", empty_lvl == 0))
        checks.append(("rows", len(tbl["rows"]) > 0))
        checks.append(("headers", len(tbl["periodHeaders"]) > 0))
        if tbl["rows"]:
            checks.append(("hdr-val", len(tbl["rows"][0]["values"]) == len(tbl["periodHeaders"])))
        table_html = extract_html_table(md_text, caption_pat)
        if table_html:
            md_labels = get_md_row_labels(table_html)
            ext_labels = [r["lineItem"].strip() for r in tbl["rows"]]
            n = min(8, len(md_labels), len(ext_labels))
            mc = sum(1 for i in range(n) if fuzzy_match(md_labels[i], ext_labels[i]))
            pct = (mc / n * 100) if n > 0 else 0
            checks.append((f"order({mc}/{n})", pct >= 75))

        p = sum(1 for _, ok in checks if ok)
        f_ = len(checks) - p
        total_pass += p
        total_fail += f_
        if f_ == 0:
            ok_count += 1
        else:
            for lbl, ok in checks:
                if not ok:
                    failures.append(f"{stem}/{fs_type}: {lbl}")

print(f"Audited {ok_count + len(failures)} statement types across 5 PDFs")
print(f"TOTAL: {total_pass} PASS, {total_fail} FAIL")
if failures:
    print(f"\nFailures:")
    for f in failures:
        print(f"  - {f}")
else:
    print("All checks passed!")

Audited 28 statement types across 5 PDFs
TOTAL: 88 PASS, 22 FAIL

Failures:
  - 10K-GOOG-01-31-2024/BalanceSheet: rows
  - 10K-GOOG-01-31-2024/BalanceSheet: order(0/0)
  - 10K-GOOG-01-31-2024/CashFlow: hdr-val
  - 10K-MSFT-07-27-2023/BalanceSheet: hdr-val
  - 10K-MSFT-07-27-2023/IncomeStatement: hdr-val
  - 10K-MSFT-07-27-2023/IncomeStatement: order(0/7)
  - 10K-MSFT-07-27-2023/ComprehensiveIncome: order(0/7)
  - 10K-MSFT-07-27-2023/Equity: hdr-val
  - 10K-MSFT-07-27-2023/Equity: order(0/5)
  - 10K-MSFT-07-27-2023/CashFlow: hdr-val
  - 10K-MSFT-07-27-2023/CashFlow: order(1/8)
  - Safehold 2024 10-K (1) 1/BalanceSheet: hdr-val
  - Safehold 2024 10-K (1) 1/IncomeStatement: hdr-val
  - Safehold 2024 10-K (1) 1/CashFlow: hdr-val
  - bnl_10K_2024/BalanceSheet: hdr-val
  - bnl_10K_2024/IncomeStatement: NOT FOUND
  - bnl_10K_2024/CashFlow: hdr-val
  - trno_10Q_Q3_2024/BalanceSheet: hdr-val
  - trno_10Q_Q3_2024/BalanceSheet: order(0/8)
  - trno_10Q_Q3_2024/IncomeStatement: hdr-val
  - trno_10Q

## Classify & Extract

Process all PDFs through the enhanced classifier.
Each PDF is sent in a single call — CU handles segmentation, classification,
and per-segment field extraction.

In [17]:
from concurrent.futures import ThreadPoolExecutor, as_completed

pdf_dir = REPO_ROOT / "email" / "attachements"
pdfs = sorted(pdf_dir.glob("*.pdf"))
print(f"Found {len(pdfs)} PDFs — processing in parallel\n")

def process_pdf(pdf_path):
    t0 = time.time()
    poller = client.begin_analyze_binary(
        analyzer_id=CLASSIFIER_ID,
        binary_input=pdf_path.read_bytes(),
        content_type="application/pdf",
    )
    result = poller.result().as_dict()
    elapsed = time.time() - t0
    contents = result.get("contents", [])
    cats = {}
    for seg in contents:
        cat = seg.get("category", "Unknown")
        cats[cat] = cats.get(cat, 0) + 1
    return pdf_path.stem, result, elapsed, cats

classifier_results = {}
t_total = time.time()

with ThreadPoolExecutor(max_workers=len(pdfs)) as executor:
    futures = {executor.submit(process_pdf, pdf): pdf for pdf in pdfs}
    for future in as_completed(futures):
        stem, result, elapsed, cats = future.result()
        classifier_results[stem] = result
        cat_str = ", ".join(f"{k}:{v}" for k, v in sorted(cats.items()))
        n_seg = sum(cats.values())
        print(f"  {stem}: {n_seg} segments, {elapsed:.0f}s  [{cat_str}]")

print(f"\nDone. Total wall time: {time.time() - t_total:.0f}s")

Found 5 PDFs — processing in parallel

  10K-GOOG-01-31-2024: 6 segments, 591s  [BalanceSheet:1, CashFlow:1, ComprehensiveIncome:1, Equity:1, IncomeStatement:1, Unknown:1]
  Safehold 2024 10-K (1) 1: 6 segments, 645s  [BalanceSheet:1, CashFlow:1, ComprehensiveIncome:1, Equity:1, IncomeStatement:1, Unknown:1]
  trno_10Q_Q3_2024: 5 segments, 712s  [BalanceSheet:1, CashFlow:1, Equity:1, IncomeStatement:1, Unknown:1]
  10K-MSFT-07-27-2023: 8 segments, 718s  [BalanceSheet:1, CashFlow:1, ComprehensiveIncome:1, Equity:3, IncomeStatement:1, Unknown:1]
  bnl_10K_2024: 5 segments, 835s  [BalanceSheet:1, CashFlow:1, Equity:1, IncomeStatement:1, Unknown:1]

Done. Total wall time: 835s


In [18]:
# Display classification and extraction results
EXPECTED_FS = ["BalanceSheet", "IncomeStatement", "ComprehensiveIncome", "Equity", "CashFlow"]

for stem, result in classifier_results.items():
    contents = result.get("contents", [])
    print(f"\n{'='*80}")
    print(f"{stem}")
    print(f"{'='*80}")

    for seg in contents:
        cat = seg.get("category", "Unknown")
        if cat == "Other":
            continue
        pg_start = seg.get("startPageNumber", "?")
        pg_end = seg.get("endPageNumber", "?")
        fields = seg.get("fields", {})
        ft = fields.get("financialTables", {})
        tables = ft.get("valueArray", [])
        print(f"\n  {cat} (pages {pg_start}-{pg_end}): {len(tables)} table(s)")
        for j, tbl_raw in enumerate(tables, 1):
            tobj = tbl_raw.get("valueObject", {})
            title = (tobj.get("tableTitle") or {}).get("valueString", "")
            stype = (tobj.get("statementType") or {}).get("valueString", "")
            rows = (tobj.get("rows") or {}).get("valueArray", [])
            hdrs = (tobj.get("periodHeaders") or {}).get("valueArray", [])
            unit = (tobj.get("unit") or {}).get("valueString", "")
            print(f"    T{j}: {stype} | {len(rows)} rows, {len(hdrs)} cols | {unit} | {title[:60]}")

    found = {seg.get("category") for seg in contents if seg.get("category") != "Other"}
    missing = [fs for fs in EXPECTED_FS if fs not in found]
    if missing:
        print(f"\n  MISSING: {', '.join(missing)}")
    else:
        print(f"\n  All 5 FS types found")


10K-GOOG-01-31-2024

  Unknown (pages 1-111): 0 table(s)

  BalanceSheet (pages 52-52): 1 table(s)
    T1: BalanceSheet | 38 rows, 2 cols |  | 

  IncomeStatement (pages 53-53): 1 table(s)
    T1: IncomeStatement | 14 rows, 3 cols | millions, except per share amounts | Alphabet Inc. CONSOLIDATED STATEMENTS OF INCOME (in millions

  ComprehensiveIncome (pages 54-54): 1 table(s)
    T1: ComprehensiveIncome | 13 rows, 3 cols | in millions | Alphabet Inc. CONSOLIDATED STATEMENTS OF COMPREHENSIVE INCOM

  Equity (pages 55-55): 1 table(s)
    T1: Equity | 24 rows, 5 cols | millions | Alphabet Inc. CONSOLIDATED STATEMENTS OF STOCKHOLDERS' EQUIT

  CashFlow (pages 56-56): 1 table(s)
    T1: CashFlow | 39 rows, 3 cols | millions | Alphabet Inc. CONSOLIDATED STATEMENTS OF CASH FLOWS (in mill

  All 5 FS types found

Safehold 2024 10-K (1) 1

  Unknown (pages 1-120): 0 table(s)

  BalanceSheet (pages 49-49): 1 table(s)
    T1: BalanceSheet | 36 rows, 2 cols | In thousands, except per share data 

## Save & Export

In [19]:
import sys
sys.path.insert(0, str(REPO_ROOT / "scripts"))
from export_to_excel import export_document

out_dir = REPO_ROOT / "output"
out_dir.mkdir(exist_ok=True)
excel_dir = out_dir / "excel"

for stem, result in classifier_results.items():
    # Save raw classifier result (includes segment metadata)
    raw_path = out_dir / f"{stem}_classified_raw.json"
    raw_path.write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")

    # Merge all FS segments' financialTables into one list for export
    merged_tables = []
    for seg in result.get("contents", []):
        if seg.get("category", "Other") == "Other":
            continue
        ft = seg.get("fields", {}).get("financialTables", {})
        merged_tables.extend(ft.get("valueArray", []))

    merged = {
        "contents": [{
            "fields": {
                "financialTables": {
                    "type": "array",
                    "valueArray": merged_tables,
                }
            }
        }]
    }
    json_path = out_dir / f"{stem}_classified.json"
    json_path.write_text(json.dumps(merged, indent=2, ensure_ascii=False), encoding="utf-8")

    # Export to Excel
    xlsx_path = export_document(json_path, excel_dir)
    print(f"{stem}: {len(merged_tables)} tables -> {json_path.name}, {xlsx_path.name}")

10K-GOOG-01-31-2024: 5 tables -> 10K-GOOG-01-31-2024_classified.json, 10K-GOOG-01-31-2024_classified.xlsx
Safehold 2024 10-K (1) 1: 5 tables -> Safehold 2024 10-K (1) 1_classified.json, Safehold 2024 10-K (1) 1_classified.xlsx
trno_10Q_Q3_2024: 5 tables -> trno_10Q_Q3_2024_classified.json, trno_10Q_Q3_2024_classified.xlsx
10K-MSFT-07-27-2023: 12 tables -> 10K-MSFT-07-27-2023_classified.json, 10K-MSFT-07-27-2023_classified.xlsx
bnl_10K_2024: 4 tables -> bnl_10K_2024_classified.json, bnl_10K_2024_classified.xlsx
